# Statistical significance checks on the pilot runs
50/100 questions, 20/50 template variations -> 1000/5000 total questions (20% of the benchmark)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser
from gsm_benchmarker.results_analyser.prompt_effect_analyser import PromptEffectAnalyser

plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
import pandas as pd
from statsmodels.formula.api import gee
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.genmod.families.family import Binomial
from statsmodels.stats.multitest import multipletests
import numpy as np

from rpy2.rinterface_lib.embedded import RRuntimeError
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# Set pandas converter as the global default for rpy2
ro.conversion.set_conversion(pandas2ri.converter + ro.default_converter)

from pymer4.models import glmer

import logging

logger = logging.getLogger('notebook')

In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()
p_standard = pp / "mini_20x50x4__14_11/final"
p_sep = pp / 'mini_sep_new__20x50__20_12/final'
p_code = pp / 'mini_code_output_20x50__05_12/final'

In [ ]:
mres_standard = MultiVariantMultiModelResultsAnalyser(p_standard)
# mres_sep = MultiVariantMultiModelResultsAnalyser(p_sep)
mres_code = MultiVariantMultiModelResultsAnalyser(p_code)

In [ ]:
# pea_sep = PromptEffectAnalyser(mres_standard, mres_sep, "Anti-babbling prompt")
pea_code = PromptEffectAnalyser(mres_standard, mres_code, "Code output prompt")

In [ ]:
models = pea_code._baseline_mres.variants['main'].models
model = models[0]
model

In [ ]:
def prep_df(variant, model_):
    res = pea_code._baseline_mres.variants[variant].full_data
    res = res[res.model == model_][:]
    res = res[['id', 'correct']]
    res['is_variant'] = [int(variant!='GSM8K')] * len(res)
    res['is_correct'] = res['correct'].astype(int)
    res.drop('correct', inplace=True, axis=1)
    return res

df = pd.concat([prep_df('GSM8K', model), prep_df('main', model)]).reset_index(drop=True)
df


In [ ]:
all_results = {}
for model in models:
    df = pd.concat([prep_df('GSM8K', model), prep_df('main', model)]).reset_index()

    glmm_model = gee("is_correct ~ is_variant", groups="id",
                     data=df, family=Binomial(),
                     cov_struct=Exchangeable())
    result = glmm_model.fit()
    all_results[model] = {'p_value': result.pvalues['is_variant']}

pd.DataFrame(all_results).T

In [ ]:
df_original = pea_code._baseline_mres.variants['GSM8K'].full_data
df_variants = pea_code._baseline_mres.variants['main'].full_data
df = pd.concat([df_original, df_variants]).reset_index(drop=True)
df

## TO DO:
- workflow: use GLMM rather than GEE (below), but otherwise - same information needed (probably - review)
- use leave-one-out for difficulty - see Clause (Pymer4 GLMM chat, towards the end, but before last question)
- both accuracies
- plot matrix, not bars, of question difficulty
- clean up, move to pea
- add multiple comparisons correction (or already added)?
- analyse global effect
- new plots
- check on the full data

In [ ]:
def fit_glmm(df):
    glmm_model = glmer(
        'correct ~ is_variant + difficulty + (1 | id)',
        data=df,
        family='binomial'
    )
    try:
        glmm_model.fit(verbose=False)  # fitting works, only getting stats fails
    except RRuntimeError:
        pass

    if glmm_model.r_model is None:
        raise RuntimeError(f"GLMM fitting failed")

    # Assign the model to an R variable first
    ro.globalenv['r_model'] = glmm_model.r_model

    # Then extract coefficients as a DataFrame
    with localconverter(ro.default_converter + pandas2ri.converter):
        coefs_df = ro.r('as.data.frame(coef(summary(r_model)))')

    res = dict(
        estimate=coefs_df.loc['is_variant', 'Estimate'],
        p_value=coefs_df.loc['is_variant', 'Pr(>|z|)'],
        std_err=coefs_df.loc['is_variant', 'Std. Error'],
    )

    return res



In [ ]:
glmm_results = []

for model_name, group_df in df.groupby('model'):
    group_df_for_glmm = pd.DataFrame({
        'id': group_df['id'].astype(int),
        'correct': group_df['correct'].astype(int),
        'is_variant': (group_df['instance'] != -1).astype(int)
    })

    difficulty = pea_code._baseline_mres.get_question_difficulty(model=model_name)
    group_df_for_glmm = group_df_for_glmm.merge(difficulty.reset_index(), on='id', how='left')

    try:
        res = fit_glmm(group_df_for_glmm)
    except RuntimeError as err:
        logger.warning(f"{model_name}: {err}")
        continue

    glmm_results.append({
        'model': model_name,
        **res,
    })

glmm_results_df = pd.DataFrame(glmm_results)

model_accuracy_drops = pea_code._baseline_mres.get_accuracy_drop_df('main').groupby('model').mean().reset_index()
glmm_results_df['accuracy_drop'] = model_accuracy_drops.accuracy_drop

# Calculate the Odds Ratio if you don't already have it
glmm_results_df['odds_ratio'] = np.exp(glmm_results_df['estimate'])

glmm_results_df['drop'] = glmm_results_df.estimate < 0

# --- 2. Calculate the 95% Confidence Intervals ---
# Calculate in log-odds space
glmm_results_df['ci_lower_log'] = glmm_results_df['estimate'] - 1.96 * glmm_results_df['std_err']
glmm_results_df['ci_upper_log'] = glmm_results_df['estimate'] + 1.96 * glmm_results_df['std_err']

# Exponentiate to get Odds Ratio CIs
glmm_results_df['ci_lower_or'] = np.exp(glmm_results_df['ci_lower_log'])
glmm_results_df['ci_upper_or'] = np.exp(glmm_results_df['ci_upper_log'])

# apply Benjamini-Hochberg procedure - controls the false discovery rate (multiple comparisons correction)
rejected, p_corrected, _, _ = multipletests(glmm_results_df['p_value'], method='fdr_bh')
glmm_results_df['p_value_corrected'] = p_corrected

glmm_results_df

In [ ]:
import seaborn as sns
from matplotlib.lines import Line2D

# --- 4. Plotting the Forest Plot ---
fig, ax = plt.subplots(figsize=(8, 6))

# Sort the dataframe so the best performing models (highest OR) are at the top
df_plot = glmm_results_df.sort_values('odds_ratio', ascending=True)

def get_color(p):
    # Using a distinct colour palette: Dark Red for strong significance,
    # Orange for moderate, and Grey for non-significant
    if p < 0.01: return '#b2182b'      # Dark Red
    elif p < 0.05: return '#ef8a62'    # Orange/Coral
    else: return '#999999'             # Grey

df_plot['color'] = df_plot['p_value'].apply(get_color)


# Loop to plot CIs and Colored Dots
for i, row in df_plot.iterrows():
    # Draw CIs (we can color these grey, or match the dot color. Grey is usually cleaner)
    ax.hlines(y=row['model'], xmin=row['ci_lower_or'], xmax=row['ci_upper_or'],
              color='darkgrey', linewidth=2, zorder=1)

    # Draw the dot using the dynamically assigned color
    ax.scatter(x=row['odds_ratio'], y=row['model'],
               color=row['color'], s=100, zorder=2, edgecolor='black', linewidth=0.5)

# Line of no effect
ax.axvline(x=1, color='black', linestyle='--', linewidth=1.2, zorder=0)

# Format X-axis
ax.set_xscale('log')
ax.set_xlabel('Odds Ratio (Log Scale)', fontsize=11)
ax.set_ylabel('LLM Model', fontsize=11)
ax.set_title('Effect of Variant Condition on LLM Performance', fontsize=14, pad=15)

# --- 4. Custom Legend ---
# Because we colored dots dynamically, we need to manually build a legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#b2182b', markersize=10, label='p < 0.01 (Strong Drop)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ef8a62', markersize=10, label='p < 0.05 (Significant Drop)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#999999', markersize=10, label='p \u2265 0.05 (Not Significant)')
]
ax.legend(handles=legend_elements, loc='lower right', title="Significance", frameon=True)

sns.despine(left=True, bottom=True)
plt.tight_layout()

In [ ]:
difficulties = {}

for model in glmm_results_df.model:
    difficulties[model] = pea_code._baseline_mres.get_question_difficulty(model=model)

# difficulties['overall'] = pea_code._baseline_mres.get_question_difficulty()

difficulties_df = pd.DataFrame(difficulties).T
difficulties_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

df_raw = difficulties_df
n_models, n_questions = df_raw.shape


# ==========================================
# 2. Calculate Marginals & Sort
# ==========================================
# Calculate overall accuracy for models and success rate for questions
model_accuracy = df_raw.sum(axis=1) / n_questions
question_accuracy = df_raw.sum(axis=0) / n_models

# Sort models from worst to best (for the y-axis)
sorted_models = model_accuracy.sort_values(ascending=True).index

# Sort questions from easiest to hardest (for the x-axis)
sorted_questions = question_accuracy.sort_values(ascending=False).index

# Reorder the raw dataframe based on the sorting logic above
df_sorted = df_raw.loc[sorted_models, sorted_questions]


# ==========================================
# 3. Plotting with GridSpec
# ==========================================
with plt.style.context('seaborn-v0_8-whitegrid'):
    # Setup the figure and the grid layout (4 rows, 5 columns)
    fig = plt.figure(figsize=(14, 8))
    gs = gridspec.GridSpec(4, 5, wspace=0.1, hspace=0.1)

    # Assign the grid spaces to axes
    ax_heatmap = fig.add_subplot(gs[1:4, 0:4])               # Bottom-left: Main heatmap
    ax_top = fig.add_subplot(gs[0:1, 0:4], sharex=ax_heatmap) # Top-left: Column marginals
    ax_right = fig.add_subplot(gs[1:4, 4:5], sharey=ax_heatmap) # Bottom-right: Row marginals

    # --- A. Center: The Heatmap ---
    cmap = sns.color_palette("mako", as_cmap=True)
    sns.heatmap(df_sorted, cmap=cmap, cbar=False, ax=ax_heatmap,
                xticklabels=False, yticklabels=True)

    ax_heatmap.set_xlabel("Questions (sorted easiest to hardest)", fontsize=12, labelpad=10)
    ax_heatmap.set_ylabel("Models (sorted worst to best)", fontsize=12, labelpad=10)

    # --- B. Top: Marginal Bar Chart for Questions (NOW COLORED) ---
    # 1. Create a normalizer that forces mapping values between 0.0 (0%) and 1.0 (100%)
    norm = plt.Normalize(0, 1)

    # 2. Pass the question accuracies through the normalizer, then into the colormap
    # This generates a list of 100 RGBA color arrays that match our heatmap theme perfectly!
    question_bar_colors = cmap(norm(question_accuracy.loc[sorted_questions]))

    # 3. Supply that list to the `color` parameter instead of a single hex string
    ax_top.bar(np.arange(n_questions) + 0.5, question_accuracy.loc[sorted_questions] * 100,
               color=question_bar_colors, width=1.0)

    ax_top.set_ylabel("% Models\nCorrect", fontsize=10)
    ax_top.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    for spine in ['top', 'right', 'bottom']:
        ax_top.spines[spine].set_visible(False)
    ax_top.grid(False)
    ax_top.set_ylim(0, 100)

# --- C. Right: Marginal Bar Chart for Models ---
    model_bar_colors = cmap(norm(model_accuracy.loc[sorted_models]))
    ax_right.barh(np.arange(n_models) + 0.5, model_accuracy.loc[sorted_models] * 100, color=model_bar_colors, height=0.8)

    ax_right.set_xlabel("% Questions Correct", fontsize=10)
    ax_right.tick_params(axis='y', which='both', left=False, labelleft=False)
    for spine in ['top', 'right', 'left']:
        ax_right.spines[spine].set_visible(False)
    ax_right.grid(False)
    ax_right.set_xlim(0, 100)

    # --- D. Final Touches ---
    fig.suptitle("Sorted Model Capabilities (Guttman Scale)", fontsize=16, fontweight='bold', y=0.95)


In [ ]:
fig, ax = plt.subplots()
ax.hist(pea_code._baseline_mres.get_question_difficulty(model=None), 21)
ax.set_xlabel("Overall question difficulty")
ax.set_ylabel("Question count")